## Futásidő-összevetés: rprev referencia (naiv K-szoros) vs. kiterjesztett több indexdátumos implementáció

Ez a notebook a szakdolgozat 5. és 7. fejezetében rögzített több indexdátumos kiterjesztéshez kapcsolódó **futásidő-mérési protokollt** valósítja meg. A mérés célja annak kvantifikálása, hogy azonos bemeneti adaton és azonos szimulációs beállítások mellett milyen számítási igénye van  
(i) a referencia-megoldásnak, ahol a $\{t_k\}_{k=1}^K$ rácsot **$K$ külön `prevalence()` függvényhívással** állítjuk elő, illetve  
(ii) a kiterjesztett rprev-implementációnak, ahol az `index` argumentum több időpont fogadására is képes, és a becslés **egyetlen eljárásfuttatásban** állítja elő a teljes prevalenciavektort és annak összefoglalóját.

### Kapcsolódás a dolgozathoz

A kiterjesztés matematikai tartalma röviden: adott $t_1 < \cdots < t_K$ rácson, az $r$-edik Monte Carlo futásban a futáson belül előállított incidens esetek $(D_{i\ell}^{(r)}, \mathbf{x}_{i\ell}^{(r)})$ és a futáson belül rögzített túlélési függvény $S^{(r)}(\cdot \mid \mathbf{x})$ alapján minden $t_k$ időpontra képezzük a prevalens státuszt, majd időpontonként összegezve kapjuk a $P^{(r)}(t_k)$ prevalenciakomponenseket és a futások feletti összefoglaló statisztikákat.

Csomagszinten a notebook azt a működést tekinti „kiterjesztett” megvalósításnak, ahol a `prevalence()` az `index` bemenetet rendezett $\texttt{index\_dates} = (t_1,\ldots,t_K)$ vektorrá normalizálja, a belső szimulációs réteg futásonként az `index_dates` rácson szolgáltat státuszinformációt, amelyből az összegző réteg időpontonként képez prevalencia-összefoglalót. A kimeneti objektum tartalmazza az `index_dates` vektort és az időpontindexelt eredményeket.

### Mérési egységek és protokoll

A dolgozat terminológiáját követve:

- **Alapkiértékelés:** a `prevalence()` egy futtatása rögzített $t$ mellett.
- **Eljárásfuttatás:** a $\{t_k\}_{k=1}^K$ időpontrács teljes kimenetének előállítása.

A naív, referencia-megoldásban egy eljárásfuttatás **$K$ darab alapkiértékelésből** áll (külön-külön időpontra), míg a kiterjesztett, natív módon több indexpontot támogató megoldásban ugyanez **egyetlen alapkiértékelésként** valósul meg.

A notebook minden paraméterkombinációra ismételt futásidő-mérést végez (`reps`), és az összidőt `system.time(...)[["elapsed"]]` alapján rögzíti. A riportolt érték minden beállításpontra azok egyes ismételt futásaiból származó eredményeinek átlaga.

A mérési protokoll kizárólag a futásidő kvantifikálására szolgál. A prevalenciabecslés pontosságának vizsgálata ettől elkülönítve, egy másik notebookban történik.

### Bemeneti adat és indexdátumok

A méréshez az rprev csomag `prevsim` példaadata szolgál alapul, melyből visszatevéses mintavétellel áll elő az $N$ soros, regiszterjellegű bemeneti tábla (a mérésben az $N$ a rekordszámot jelöli). Az indexdátumok egy rögzített kezdődátumtól (`index_start`) fix naplépéssel (`index_step_days`) generált, rendezett $K$ hosszú sorozatot alkotnak.

A notebookban az indexdátumok generálása determinisztikus: adott $K$, `index_start` és `index_step_days` mellett az `index_dates` teljesen reprodukálható, és a különböző mérési pontok között kizárólag a konfigurációs paraméterek változnak.

A bemeneti táblát a futásidő skálázódásának vizsgálata motiválja, ezért az adat-előállítás során nem cél egy valós regiszter teljes, esetszintű konzisztenciájának biztosítása. A visszatevéses mintavétel következtében előfordulhatnak ismétlődő rekordok, ezért a kapott állomány nem tekintendő tiszta regiszternek a szokásos adatminőségi kritériumok szerint. A mérés érvényességét viszont ez nem érinti, mivel a vizsgálat tárgya kizárólag a számítási terhelés és annak $N$ és $K$ szerinti alakulása.

### Szimulációs és bootstrap beállítások

A mérések rácsa a következő paraméterek mentén szervezett:

- $N$: a bemeneti tábla sorainak száma (adatméret, rekordszám),
- $K$: az indexdátumok száma,
- `N_boot`: a `prevalence()` hívásban használt bootstrap ismétlések száma,
- `reps`: az időmérés ismétlésszáma paraméterpontonként.

A fenti rács mellett a hívásokban több beállítás rögzített, és a futtatások között nem változik: az indexdátum-generálás paraméterei, valamint a túlélési eloszlás választása.

### Kimenet

A futás végén egy eredménytábla készül, amely minden paraméterpontra tartalmazza:

- `N`, `K`, `N_boot`,
- `reps`,
- `elapsed_sec_mean`: az eljárásfuttatás átlagos futásideje másodpercben.

---

#### 1. Környezet és példaadat betöltése
- A szükséges csomagok betöltése és az `rprev` csomag `prevsim` példaadatának beolvasása egységes bemenetként.
- A `base_df` tábla szolgál alapul a későbbi, kontrollált méretű bemeneti állományok előállításához.

In [ ]:
# Csomagok betöltése
suppressPackageStartupMessages({
  library(rprev)
  library(survival)
})

# Alapadat betöltése
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)

#### 2. Közös paraméter konfiguráció
- A futásidő-mérés paraméterrácsának és a rögzített modellspecifikációs beállításoknak a központi definiálása a reprodukálhatóság érdekében.
- A referencia és a kiterjesztett megoldása futtatása ugyanebből a paraméterkészletből dolgozik, így az összevetés konzisztens.

In [ ]:
# ---- Közös benchmark beállítások (baseline + kiterjesztett összehasonlítás) ----
COMMON_CFG <- list(
  N_values              = c(1e2, 1e3),
  K_values              = c(1, 5, 20),
  N_boot_values         = c(100, 200),
  reps                  = 3L,
  num_years_to_estimate = c(15),
  index_start           = as.Date("2013-01-30"),
  index_step_days       = 30L,
  dist                  = "weibull",
  population_size       = 1e6
)

# COMMON_CFG <- list(
#   N_values              = c(1e2, 1e3),
#   K_values              = c(1, 10, 20. 30),
#   N_boot_values         = c(100, 500, 1000),
#   reps                  = 3L,
#   num_years_to_estimate = c(15),
#   index_start           = as.Date("2013-01-30"),
#   index_step_days       = 30L,
#   dist                  = "weibull",
#   population_size       = 1e6
# )


In [30]:
# Skálázott adatok
scaled_data <- setNames(
  lapply(COMMON_CFG$N_values, function(N) {
    set.seed(10000L + as.integer(N))
    base_df[sample.int(nrow(base_df), size = as.integer(N), replace = TRUE), , drop = FALSE]
  }),
  paste0("N", COMMON_CFG$N_values)
)

scaled_data_n <- data.frame(
  N = COMMON_CFG$N_values,
  n_rows = vapply(scaled_data, nrow, integer(1)),
  stringsAsFactors = FALSE
)
print(scaled_data_n)


#### 3. Referencia-megoldás futásidő-mérése: K külön `prevalence()` hívás
- A segédfüggvények előállítják az $N$ soros bemenetet és a $K$ darab indexdátumot, majd a referencia-megoldás $K$ külön hívással elvégzi a számítást.
- Az időmérés `system.time()` alapján történik, és paraméterpontonként az `reps` ismétlésből származó futásidők kerülnek átlagolásra .

In [ ]:
# ---- Segédfüggvény: N soros "registry" a prevsim-ből ----
make_registry <- function(N, seed = 1L) {
  set.seed(as.integer(seed))
  base_df[sample.int(nrow(base_df), size = as.integer(N), replace = TRUE), , drop = FALSE]
}

# ---- Segédfüggvény: K db indexdátum (Date) ----
make_index_dates <- function(K, start_date, step_days) {
  seq.Date(from = start_date, by = as.integer(step_days), length.out = as.integer(K))
}

# ---- Egy futás: 1 indexdátumra ----
# index_date_chr: "YYYY-MM-DD"
run_rprev_once <- function(dat, index_date_chr, N_boot) {
  rprev::prevalence(
    index = index_date_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

# ---- Mérés: K-szor hívjuk a prevalence()-t, és mérjük az összidőt ----
measure_one_setting <- function(N, K, N_boot, seed = 1L) {
  dat <- make_registry(N, seed = seed)

  idx_dates <- make_index_dates(K, COMMON_CFG$index_start, COMMON_CFG$index_step_days)

  # KRITIKUS FIX:
  # for(d in idx_dates) közben a Date class le tud esni -> előre karakterré alakítjuk.
  idx_dates_chr <- format(idx_dates, "%Y-%m-%d")

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()
    t <- system.time({
      for (d_chr in idx_dates_chr) {
        invisible(run_rprev_once(dat, d_chr, N_boot))
      }
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

# ---- Grid futtatás ----
results <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
          measure_one_setting(
            N = N,
            K = K,
            N_boot = N_boot,
            seed = 1000 + as.integer(N) + as.integer(K) + as.integer(N_boot)
          )
      }))
    }))
  })
)

results <- results[order(results$N, results$K, results$N_boot), ]
within(results, { elapsed_sec_mean <- round(elapsed_sec_mean, 2) })

# write.csv(results, "rprev_runtime_pilot.csv", row.names = FALSE)

Run: N=100, K=1, N_boot=100

Run: N=100, K=1, N_boot=200

Run: N=100, K=5, N_boot=100

Run: N=100, K=5, N_boot=200

Run: N=100, K=20, N_boot=100

Run: N=100, K=20, N_boot=200

Run: N=1000, K=1, N_boot=100

Run: N=1000, K=1, N_boot=200

Run: N=1000, K=5, N_boot=100

Run: N=1000, K=5, N_boot=200

Run: N=1000, K=20, N_boot=100

Run: N=1000, K=20, N_boot=200



      N  K N_boot reps elapsed_sec_mean
1   100  1    100    3         1.270000
2   100  1    200    3         1.956667
3   100  5    100    3         5.066667
4   100  5    200    3        14.723333
5   100 20    100    3        28.216667
6   100 20    200    3        54.556667
7  1000  1    100    3         2.346667
8  1000  1    200    3         4.533333
9  1000  5    100    3        11.036667
10 1000  5    200    3        22.303333
11 1000 20    100    3        44.223333
12 1000 20    200    3       100.103333


#### 4. Kiterjesztett rprev csomag betöltése lokális forrásból és git-meta rögzítése
- A csomag lokális forrásból kerül betöltésre, így a futtatás ténylegesen a vizsgált, módosított implementációval történik.
- A futtatott verzió azonosíthatóságát a git ág és commit azonosító kiírása biztosítja, ami az eredmények dokumentálásához szükséges.

In [26]:
# Projekt gyökérkönyvtárának meghatározása
project_root <- trimws(system2("git", c("rev-parse", "--show-toplevel"), stdout = TRUE))
if (length(project_root) == 0 || !dir.exists(project_root)) {
  stop("Could not resolve project root from git.")
}

# A csomag lokális forráskódjának betöltése
if ("package:rprev" %in% search()) detach("package:rprev", unload = TRUE, character.only = TRUE)
if ("rprev" %in% loadedNamespaces()) unloadNamespace("rprev")
if (!requireNamespace("devtools", quietly = TRUE)) stop("devtools package is not available.")

suppressPackageStartupMessages(devtools::load_all(project_root, quiet = TRUE))

cat(
  "Loaded rprev from: ",
  normalizePath(getNamespaceInfo("rprev", "path"), winslash = "/"),
  "\n",
  sep = ""
)

Loaded rprev from: C:/Users/600972868/OneDrive - BT Plc/Desktop/Sajat Mappa/00 - BGE/Alkalmazott Matematika/Szakdolgozat/repo/rprev-ext


In [29]:
# Git meta (branch + commit) kiírása (Windows-kompatibilis: setwd, nincs -C)

git_cmd <- function(args) {
  out <- suppressWarnings(
    tryCatch(system2("git", args, stdout = TRUE, stderr = TRUE), error = function(e) character(0))
  )
  status <- attr(out, "status")
  if (!is.null(status) || length(out) == 0) {
    return(NA_character_)
  }
  trimws(out[[1]])
}

old_wd <- getwd()
on.exit(setwd(old_wd), add = TRUE)
setwd(project_root)

branch <- git_cmd(c("rev-parse", "--abbrev-ref", "HEAD"))
commit <- git_cmd(c("rev-parse", "HEAD"))

cat("Git branch: ", branch, "\n", sep = "")
cat("Git commit: ", commit, "\n", sep = "")

Git branch: notebooks/runtime-benchmarks
Git commit: 03d56725df9eef382aeb83ef63bd81112ac50fa3


#### 5. Kiterjesztett megoldás futásidő-mérése: natív több indexdátumos `prevalence()` hívás
- Az eljárás egyetlen `prevalence()` hívásban kapja meg a $K$ darab indexdátumot, és ugyanebben a futtatásban állítja elő a teljes kimenetet.
- Az időmérés azonos módon, `system.time()` alapján történik, és `reps` ismétlés átlaga kerül riportolásra.

In [ ]:
# Kimenet formátuma megegyezik a baseline táblával.

run_rprev_ext_once <- function(dat, index_dates_chr, N_boot) {
  rprev::prevalence(
    index = index_dates_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

measure_one_setting_ext <- function(N, K, N_boot, seed = 1L) {
  dat <- make_registry(N, seed = seed)
  idx_dates <- make_index_dates(K, COMMON_CFG$index_start, COMMON_CFG$index_step_days)
  idx_dates_chr <- format(idx_dates, "%Y-%m-%d")

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()
    t <- system.time({
      invisible(run_rprev_ext_once(dat, idx_dates_chr, N_boot))
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

results_ext <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Ext run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
          measure_one_setting_ext(
            N = N,
            K = K,
            N_boot = N_boot,
            seed = 2000 + as.integer(N) + as.integer(K) + as.integer(N_boot)
          )
      }))
    }))
  })
)

results_ext <- results_ext[order(results_ext$N, results_ext$K, results_ext$N_boot), ]
within(results_ext, { elapsed_sec_mean <- round(elapsed_sec_mean, 2) })


Ext run: N=100, K=1, N_boot=100



Ext run: N=100, K=1, N_boot=200

Ext run: N=100, K=5, N_boot=100

Ext run: N=100, K=5, N_boot=200

Ext run: N=100, K=20, N_boot=100

Ext run: N=100, K=20, N_boot=200

Ext run: N=1000, K=1, N_boot=100

Ext run: N=1000, K=1, N_boot=200

Ext run: N=1000, K=5, N_boot=100

Ext run: N=1000, K=5, N_boot=200

Ext run: N=1000, K=20, N_boot=100

Ext run: N=1000, K=20, N_boot=200



      N  K N_boot reps elapsed_sec_mean
1   100  1    100    3         2.423333
2   100  1    200    3         4.716667
3   100  5    100    3         3.670000
4   100  5    200    3         7.306667
5   100 20    100    3        12.753333
6   100 20    200    3        17.593333
7  1000  1    100    3         2.213333
8  1000  1    200    3         4.586667
9  1000  5    100    3         4.453333
10 1000  5    200    3         9.316667
11 1000 20    100    3        14.376667
12 1000 20    200    3        26.020000


#### 6. Összesítés: referencia- és kiterjesztett futásidők összevetése
- A két eredménytábla paraméterpontonként összeillesztésre kerül, így közvetlenül összehasonlítható a referencia- és a kiterjesztett megoldás átlagos futásideje.
- A különbség abszolút (`delta_sec`) és relatív (`delta_pct`) formában is kiszámításra kerül, ami a gyorsulás mértékét számszerűsíti.

In [22]:
# ---- Összesítő tábla: baseline vs natív multi-index ----
summary_runtime <- merge(
  results[, c("N", "K", "N_boot", "reps", "elapsed_sec_mean")],
  results_ext[, c("N", "K", "N_boot", "reps", "elapsed_sec_mean")],
  by = c("N", "K", "N_boot", "reps"),
  suffixes = c("_baseline", "_multi")
)

summary_runtime <- summary_runtime[order(summary_runtime$N, summary_runtime$K, summary_runtime$N_boot), ]
summary_runtime$delta_sec <- summary_runtime$elapsed_sec_mean_multi - summary_runtime$elapsed_sec_mean_baseline
summary_runtime$delta_pct <- 100 * summary_runtime$delta_sec / summary_runtime$elapsed_sec_mean_baseline

summary_runtime_disp <- within(summary_runtime, {
  elapsed_sec_mean_baseline <- round(elapsed_sec_mean_baseline, 2)
  elapsed_sec_mean_multi <- round(elapsed_sec_mean_multi, 2)
  delta_sec <- round(delta_sec, 2)
  delta_pct <- round(delta_pct, 2)
})
summary_runtime_disp


,N,K,N_boot,reps,elapsed_sec_mean_baseline,elapsed_sec_mean_multi,delta_sec,delta_pct
,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
1,100,1,100,3,1.27,2.42,1.15,90.81
2,100,1,200,3,1.96,4.72,2.76,141.06
5,100,5,100,3,5.07,3.67,-1.40,-27.57
6,100,5,200,3,14.72,7.31,-7.42,-50.37
3,100,20,100,3,28.22,12.75,-15.46,-54.80
4,100,20,200,3,54.56,17.59,-36.96,-67.75
7,1000,1,100,3,2.35,2.21,-0.13,-5.68
8,1000,1,200,3,4.53,4.59,0.05,1.18
11,1000,5,100,3,11.04,4.45,-6.58,-59.65
